# 05 — Streamlit e Deployment do Modelo

## Objetivo

Nesta etapa, o modelo de risco de crédito desenvolvido nas fases anteriores será transformado em uma aplicação utilizável por um usuário final.

Até este ponto, o projeto passou pelas etapas de:

1. entendimento dos dados;
2. preparação dos dados;
3. modelagem;
4. avaliação dos modelos;
5. definição do limiar operacional;
6. interpretação das previsões com SHAP;
7. exportação do modelo treinado.

O próximo objetivo é disponibilizar esse modelo por meio de uma aplicação web desenvolvida com **Streamlit**.

A aplicação permitirá que um analista informe os dados de um novo cliente e obtenha:

- a probabilidade estimada de inadimplência;
- a classificação produzida pelo modelo;
- a decisão baseada no limiar operacional definido no projeto;
- os principais fatores que influenciaram a previsão daquele cliente por meio de SHAP.

O modelo utilizado será o **XGBoost já treinado e exportado**, sem realizar novo treinamento durante a execução da aplicação.

## 1. O que é Streamlit?

Streamlit é um framework Python utilizado para construir aplicações web interativas, especialmente útil em projetos de Ciência de Dados e Machine Learning.

Ele permite transformar código Python em uma interface gráfica acessível pelo navegador sem a necessidade de desenvolver toda a aplicação utilizando diretamente HTML, CSS e JavaScript.

No nosso projeto, o Streamlit funcionará como a **interface entre o usuário e o modelo de Machine Learning**.

O Streamlit não realiza a previsão. A previsão continua sendo responsabilidade do pipeline e do modelo XGBoost.

O fluxo será:

**Usuário → Streamlit → Pipeline → XGBoost → Probabilidade → Decisão → Streamlit**

Assim, o analista poderá utilizar o modelo sem precisar executar notebooks ou escrever código Python.

## 2. Arquitetura da aplicação

A aplicação será composta por diferentes elementos que já foram desenvolvidos durante o projeto.

### Streamlit

Responsável pela interface com o usuário.

Receberá os dados do cliente e apresentará os resultados da análise.

### Pipeline

O pipeline salvo contém as etapas necessárias para transformar os dados de entrada e realizar a previsão:

**Dados originais → Preparação → Imputação → XGBoost**

Isso garante que novos clientes passem pelas mesmas transformações utilizadas durante o desenvolvimento do modelo.

### XGBoost

É o modelo responsável por estimar a probabilidade de inadimplência.

### Limiar operacional

A análise de custo realizada anteriormente definiu para este projeto o limiar:

**0,10 (10%)**

Portanto:

- probabilidade < 10% → classe 0;
- probabilidade ≥ 10% → classe 1.

O limiar foi armazenado juntamente com o pipeline no artefato de deploy.

### SHAP

Será utilizado para explicar individualmente as previsões.

Além de apresentar a probabilidade e a decisão, a aplicação poderá mostrar quais características do cliente tiveram maior influência sobre aquela previsão específica.

## 3. O papel do arquivo `app.py`

A aplicação Streamlit será implementada em um arquivo Python chamado:

`app.py`

Diferentemente dos notebooks, que são utilizados principalmente para exploração, experimentação, documentação e desenvolvimento, o `app.py` será o programa responsável pela execução da aplicação.

Nele serão definidos:

- configuração e aparência da página;
- carregamento do modelo salvo;
- campos para entrada dos dados do cliente;
- organização desses dados no formato esperado pelo pipeline;
- cálculo da probabilidade de inadimplência;
- aplicação do limiar operacional;
- apresentação da decisão;
- análise SHAP individual.

O `app.py` não treinará novamente o modelo.

Ele apenas carregará o artefato previamente treinado e salvo em:

`models/modelo_credito_xgb.joblib`

Portanto, existe uma separação entre:

**Desenvolvimento do modelo → notebooks**

e

**Utilização do modelo → app.py**

## 4. Servidor e localhost

Uma aplicação Streamlit é acessada por meio de um navegador, mas o código continua sendo executado em Python.

Quando executamos:

`uv run streamlit run app.py`

o Streamlit inicia um servidor web.

Durante o desenvolvimento, esse servidor é executado no próprio computador.

Por isso, o Streamlit normalmente disponibiliza um endereço semelhante a:

`http://localhost:8501`

### O que significa localhost?

`localhost` representa o próprio computador onde a aplicação está sendo executada.

Nesse momento, o fluxo é:

**app.py → Streamlit → servidor local → navegador**

O navegador apresenta a interface, enquanto o código Python, o pipeline e o modelo continuam sendo executados pelo ambiente do projeto.

A porta `8501` é normalmente utilizada pelo Streamlit para disponibilizar a aplicação localmente.

Portanto, executar a aplicação em `localhost` ainda não significa que ela foi publicada na internet.

## 5. Aplicação local e deploy

Durante o desenvolvimento, a aplicação será executada localmente:

**Usuário → navegador → localhost → Streamlit → modelo**

Isso permite desenvolver e testar todo o sistema antes de disponibilizá-lo para outras pessoas.

Depois que a aplicação estiver validada, poderá ser realizado o deploy em um servidor acessível pela internet.

O fluxo passa então a ser semelhante a:

**Usuário → internet → servidor → aplicação Streamlit → modelo**

O servidor precisará possuir:

- o código da aplicação;
- as dependências Python;
- o código de pré-processamento;
- o modelo treinado;
- as configurações necessárias para executar o projeto.

Por isso, ferramentas utilizadas anteriormente no projeto passam a ter importância também no deployment:

- **Git:** controla as versões do código;
- **GitHub:** pode armazenar o repositório remotamente;
- **uv:** registra e reproduz as dependências do ambiente;
- **Streamlit:** executa a aplicação web;
- **joblib:** permite carregar o modelo previamente treinado.

## 6. O artefato do modelo

O modelo treinado foi serializado utilizando `joblib`.

O arquivo criado foi:

`models/modelo_credito_xgb.joblib`

Serializar significa transformar objetos Python em um formato que possa ser armazenado em disco e posteriormente reconstruído.

Neste projeto, o artefato contém:

- o pipeline completo;
- o modelo XGBoost treinado;
- o limiar operacional;
- a identificação do modelo;
- a definição da classe positiva.

Isso permite separar dois momentos:

### Treinamento

O modelo aprende utilizando os dados de treino e posteriormente é salvo.

### Inferência

A aplicação carrega o modelo já treinado e o utiliza para realizar previsões para novos clientes.

Assim, o Streamlit não precisa treinar o XGBoost sempre que um usuário realiza uma análise de crédito.

Ele apenas carrega o conhecimento que já foi aprendido e armazenado no artefato.

```text
    TREINAMENTO
    dados históricos
        ↓
    XGBoost aprende
        ↓
    modelo_credito_xgb.joblib


    INFERÊNCIA
    novo cliente
        ↓
    modelo_credito_xgb.joblib
        ↓
    probabilidade de inadimplência
```

Essa diferença entre treinamento e inferência é central quando saímos do notebook e vamos para produção.

## 7. Entrada de um novo cliente

A aplicação Streamlit deverá solicitar apenas as informações originais disponíveis para um novo cliente.

Neste projeto, o formulário receberá 10 variáveis:

1. `idade`
2. `renda_mensal`
3. `dependentes`
4. `uso_limite_rotativo`
5. `razao_divida`
6. `linhas_credito_abertas`
7. `financiamentos_imobiliarios`
8. `atrasos_30_59_dias`
9. `atrasos_60_89_dias`
10. `atrasos_90_mais_dias`

Essas informações serão organizadas em um DataFrame com uma única linha, representando o cliente que está sendo analisado.

Exemplo conceitual:

| idade | renda_mensal | dependentes | uso_limite_rotativo | ... |
|---:|---:|---:|---:|---|
| 35 | 5000 | 0 | 0,30 | ... |

O usuário não precisará informar as variáveis derivadas criadas durante a preparação dos dados.

Essas transformações serão realizadas automaticamente pelo pipeline antes que os dados sejam enviados ao XGBoost.

## 8. Pré-processamento em produção

Durante o desenvolvimento do projeto foi criada a função `preparar_dados_credito`, localizada em:

`src/consumer_credit_risk/preprocessing.py`

Essa função faz parte do pipeline salvo juntamente com o modelo.

Assim, quando a aplicação fornecer as 10 características originais de um cliente, o pipeline aplicará automaticamente as mesmas regras de preparação utilizadas durante o desenvolvimento.

O fluxo será:

**10 variáveis originais** \
↓ \
**preparação dos dados** \
↓ \
**criação e tratamento das características** \
↓ \
**imputação de valores ausentes** \
↓ \
**15 variáveis preparadas** \
↓ \
**XGBoost** 

Isso é importante porque o modelo deve receber os novos dados no mesmo formato utilizado durante seu treinamento.

O formulário do Streamlit, portanto, não precisa reproduzir manualmente as transformações realizadas durante a preparação dos dados.

Essa responsabilidade pertence ao pipeline.

## 9. Probabilidade de inadimplência

Depois que os dados do cliente forem preparados, o XGBoost calculará a probabilidade estimada de pertencimento à classe positiva.

Neste projeto:

- classe `0` = não inadimplente;
- classe `1` = inadimplente.

A aplicação utilizará:

`predict_proba()`

Esse método retorna as probabilidades estimadas para cada classe.

Conceitualmente:

`[probabilidade_classe_0, probabilidade_classe_1]`

Como o interesse da aplicação é estimar o risco de inadimplência, será utilizada a probabilidade correspondente à classe `1`.

Exemplo:

`[0.93, 0.07]`

significa aproximadamente:

- 93% de probabilidade estimada para a classe 0;
- 7% de probabilidade estimada para a classe 1.

Nesse exemplo, a probabilidade de inadimplência apresentada pela aplicação seria:

**7%**

## 10. Probabilidade e decisão são conceitos diferentes

O XGBoost produz inicialmente uma **probabilidade**.

Por exemplo:

**Probabilidade de inadimplência = 8,6%**

Para transformar essa probabilidade em uma classificação, é necessário utilizar um **limiar de decisão**.

O valor padrão utilizado frequentemente em problemas de classificação é 0,50. Entretanto, esse valor não precisa ser utilizado obrigatoriamente.

Neste projeto, os custos dos erros são diferentes:

- Falso Negativo (FN): R$ 5.000;
- Falso Positivo (FP): R$ 500.

Como um falso negativo possui custo dez vezes maior, foi realizada uma análise específica para encontrar o limiar que minimizava o custo total.

Para o XGBoost utilizado neste projeto, o menor custo entre os limiares avaliados ocorreu em:

**limiar = 0,10**

Portanto, a aplicação utilizará:

**probabilidade < 0,10 → classe 0**

**probabilidade ≥ 0,10 → classe 1**

Assim, a probabilidade é uma saída produzida pelo modelo, enquanto a decisão depende também da regra operacional definida para utilização dessa probabilidade.

Esse ponto é muito importante. Nosso app não fará simplesmente:

```text
    XGBoost → APROVA/RECUSA
```
Na realidade será:

```text
                    XGBoost
                       ↓
             probabilidade = 8,6%
                       ↓
                limiar = 10%
                       ↓
                    8,6 < 10
                       ↓
                    classe 0
```                

## 11. Explicação individual com SHAP

Além da probabilidade e da classificação, a aplicação deverá explicar quais características tiveram maior influência sobre a previsão daquele cliente.

Essa análise será realizada utilizando SHAP.

Na etapa anterior do projeto, o SHAP foi utilizado para analisar globalmente quais variáveis eram mais importantes para o XGBoost.

No deploy, o objetivo será diferente.

Será realizada uma análise **individual**, considerando apenas o cliente que acabou de ser avaliado.

O fluxo será:

**Dados do cliente** \
↓ \
**Pipeline** \
↓ \
**XGBoost** \
↓ \
**Probabilidade** \
↓ \
**Decisão** \
↓ \
**SHAP individual**

A aplicação poderá apresentar, por exemplo, os cinco fatores que mais influenciaram aquela previsão.

É importante distinguir:

- **SHAP global:** explica o comportamento geral do modelo;
- **SHAP individual:** explica uma previsão específica.

Assim, dois clientes com a mesma classificação podem possuir razões diferentes para o resultado produzido pelo modelo.

## 12. Estrutura do projeto para o deployment

A aplicação será desenvolvida dentro do mesmo projeto utilizado para construir o modelo.

A estrutura principal será:

    consumer_credit_risk/
    │
    ├── app.py
    │
    ├── models/
    │   └── modelo_credito_xgb.joblib
    │
    ├── src/
    │   └── consumer_credit_risk/
    │       ├── __init__.py
    │       ├── data.py
    │       └── preprocessing.py
    │
    ├── notebooks/
    │   ├── 01_entendimento_dados.ipynb
    │   ├── 02_preparacao_dados.ipynb
    │   ├── 03_modelagem.ipynb
    │   ├── 04_avaliacao_deploy.ipynb
    │   └── 05_streamlit_deploy.ipynb
    │
    ├── pyproject.toml
    └── uv.lock

### Responsabilidade dos principais componentes

**app.py**

Será o ponto de entrada da aplicação Streamlit. Ele receberá os dados do usuário, carregará o modelo, realizará a inferência e apresentará os resultados.

**models/modelo_credito_xgb.joblib**

Contém o pipeline XGBoost já treinado e o limiar operacional definido durante a avaliação.

**preprocessing.py**

Contém a função utilizada pelo pipeline para preparar os dados antes que eles sejam enviados ao modelo.

**pyproject.toml**

Registra as principais configurações e dependências Python do projeto.

**uv.lock**

Registra versões específicas das dependências, contribuindo para a reprodutibilidade do ambiente.

## 13. Construção da aplicação

O primeiro objetivo será criar uma versão mínima da aplicação.

Antes de construir formulários, previsões e explicações SHAP, será verificado se o Streamlit consegue:

1. iniciar a aplicação;
2. localizar o artefato salvo;
3. carregar o pipeline;
4. recuperar o limiar operacional;
5. apresentar essas informações no navegador.

Essa abordagem incremental facilita a identificação de erros.

Somente depois dessa validação serão adicionados os campos de entrada dos clientes e a lógica de previsão.

Agora saímos momentaneamente do notebook.

**Crie app.py**

No Explorer do VS Code, clique com o botão direito sobre a **raiz** `consumer_credit_risk`, não sobre `notebooks`.

Crie:
```text
app.py
```
Sua estrutura ficará:

```text
    consumer_credit_risk/
    ├── app.py          ← novo
    ├── data/
    ├── models/
    ├── notebooks/
    ├── reports/
    ├── src/
...
```
**Primeiro código do app.py**

```python
import joblib
import streamlit as st
from pathlib import Path


# ============================================================
# Configuração da página
# ============================================================

st.set_page_config(
    page_title="Análise de Risco de Crédito",
    page_icon="💳",
    layout="wide",
)


# ============================================================
# Caminhos do projeto
# ============================================================

PROJECT_ROOT = Path(__file__).resolve().parent

MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "modelo_credito_xgb.joblib"
)


# ============================================================
# Carregamento do artefato
# ============================================================

artefato = joblib.load(MODEL_PATH)

pipeline = artefato["pipeline"]
limiar = artefato["limiar"]
nome_modelo = artefato["modelo"]


# ============================================================
# Interface
# ============================================================

st.title("💳 Análise de Risco de Crédito")

st.write(
    f"Sistema baseado no modelo **{nome_modelo}** "
    f"(Limiar operacional: **{limiar:.0%}**)"
)

st.success("Modelo carregado com sucesso.")

```

## 14. Primeira execução local da aplicação

A primeira versão da aplicação foi executada com sucesso utilizando:

`uv run streamlit run app.py`

O Streamlit iniciou um servidor local na porta 8501, permitindo acessar a aplicação pelo navegador através de:

`http://localhost:8501`

Nesta primeira execução foram validados:

- funcionamento do Streamlit;
- execução correta do `app.py`;
- localização do arquivo `modelo_credito_xgb.joblib`;
- carregamento do artefato com `joblib`;
- recuperação do pipeline treinado;
- recuperação do modelo XGBoost;
- recuperação do limiar operacional de 0,10.

A aplicação ainda não realiza previsões para novos clientes.

Nesta etapa, o objetivo foi apenas validar a comunicação entre:

**Navegador → Streamlit → app.py → artefato do modelo**

Com essa infraestrutura funcionando, a próxima etapa será construir o formulário para entrada dos dados de um novo cliente.

## 15. Widgets de entrada do Streamlit

O Streamlit possui componentes interativos chamados **widgets**.

Widgets permitem que o usuário forneça informações para o programa por meio da interface gráfica.

Um exemplo é:

`st.number_input()`

Esse widget cria um campo para entrada de valores numéricos.

Exemplo conceitual:

`idade = st.number_input("Idade")`

Nesse caso:

1. o Streamlit apresenta um campo chamado `Idade`;
2. o usuário informa um valor;
3. o valor informado é armazenado na variável Python `idade`.

Assim, a interface gráfica e o código Python estão diretamente conectados.

Outros exemplos de widgets incluem:

- `st.text_input()` — entrada de texto;
- `st.number_input()` — entrada numérica;
- `st.selectbox()` — seleção entre opções;
- `st.checkbox()` — seleção verdadeiro/falso;
- `st.button()` — botão de ação.

Neste projeto, a maior parte das informações do cliente será coletada utilizando `st.number_input()`.

## 16. Organização da interface com colunas

O Streamlit permite dividir horizontalmente a página utilizando `st.columns()`.

Por exemplo:

`col1, col2 = st.columns(2)`

cria duas regiões lado a lado.

Depois podemos determinar em qual coluna cada elemento será apresentado.

Conceitualmente:

| Coluna 1 | Coluna 2 |
|---|---|
| Idade | Linhas de crédito abertas |
| Renda mensal | Financiamentos imobiliários |
| Dependentes | Atrasos de 30–59 dias |
| Uso do limite rotativo | Atrasos de 60–89 dias |
| Razão da dívida | Atrasos de 90+ dias |

Essa organização melhora a utilização do espaço disponível e aproxima a interface do formato desejado para a aplicação.

## 17. Formulários no Streamlit

Os dados do cliente serão agrupados utilizando um formulário do Streamlit.

Um formulário pode ser criado com:

`st.form()`

Dentro dele são colocados os campos que o usuário deverá preencher.

Ao final, utilizamos:

`st.form_submit_button()`

Isso permite que o usuário preencha todas as informações e somente depois solicite a análise.

O fluxo será:

**Preencher dados → clicar em "Realizar Análise de Crédito" → executar previsão**

Essa abordagem é adequada porque não queremos realizar uma nova previsão a cada alteração individual de um campo.

A previsão será executada somente quando o formulário for enviado.

## 18. Transformando os dados do formulário em um DataFrame

Os valores preenchidos no formulário são armazenados em variáveis Python.

Entretanto, o pipeline foi desenvolvido para receber dados estruturados com as mesmas colunas utilizadas durante o treinamento.

Por isso, quando o usuário clicar em **Realizar Análise de Crédito**, os valores do formulário serão organizados em um `DataFrame` do pandas contendo uma única linha.

Essa linha representa um novo cliente.

O fluxo passa a ser:

**Formulário Streamlit** \
↓ \
**Variáveis Python** \
↓ \
**DataFrame com 1 cliente e 10 características** \
↓ \
**Pipeline** \
↓ \
**Preparação dos dados** \
↓ \
**Imputação** \
↓ \
**XGBoost** \
↓ \
**Probabilidade de inadimplência**

É fundamental que os nomes das colunas fornecidas ao pipeline sejam os mesmos esperados durante o desenvolvimento do modelo.

## 19. Primeira inferência na aplicação

Após o envio do formulário, os dados do novo cliente são organizados em um DataFrame contendo uma linha e as 10 características originais esperadas pelo pipeline.

A partir desse DataFrame, podemos realizar a inferência utilizando o pipeline treinado.

Para obter a probabilidade de inadimplência será utilizado:

`pipeline.predict_proba(cliente)`

O método retorna a probabilidade estimada para cada classe.

Como a classe positiva do projeto é:

`1 = inadimplente`

utilizamos a probabilidade correspondente à classe 1.

Conceitualmente:

**Novo cliente** \
↓ \
**DataFrame (1 × 10)** \
↓\
**Pipeline**\
↓\
**Preparação**\
↓\
**Imputação**\
↓\
**XGBoost**\
↓\
**predict_proba()**\
↓\
**Probabilidade de inadimplência**\

Depois, essa probabilidade será comparada ao limiar operacional armazenado no artefato.

Neste projeto:

**limiar = 0,10**

Portanto:

- probabilidade < 0,10 → classe 0;
- probabilidade ≥ 0,10 → classe 1.

## 20. Validação da primeira previsão no Streamlit

A aplicação realizou com sucesso sua primeira inferência completa.

Utilizando os valores iniciais definidos no formulário, o modelo XGBoost estimou:

**Probabilidade de inadimplência: 2,10%**

O limiar operacional armazenado no artefato é:

**10,00%**

Como:

**2,10% < 10,00%**

a aplicação classificou o cliente como classe `0` e apresentou a decisão:

**CRÉDITO APROVADO**

Essa execução confirmou o funcionamento integrado de:

**Formulário → DataFrame → Pipeline → Pré-processamento → Imputação → XGBoost → Probabilidade → Limiar → Decisão**

É importante observar que a decisão não utiliza o limiar padrão de 50% do classificador.

A aplicação utiliza explicitamente o limiar operacional de 10%, definido anteriormente por meio da análise dos custos de Falsos Negativos e Falsos Positivos.

## 21. SHAP individual na aplicação

Depois de calcular a probabilidade de inadimplência e aplicar o limiar operacional, a aplicação deverá explicar a previsão daquele cliente específico.

Para isso será utilizado o SHAP.

Nesta etapa, não queremos medir a importância média das variáveis para todos os clientes. Queremos explicar apenas a previsão realizada para o cliente informado no formulário.

Para calcular o SHAP individual será necessário:

1. receber as 10 variáveis originais do cliente;
2. aplicar as etapas de preparação e imputação do pipeline;
3. obter as 15 características efetivamente utilizadas pelo XGBoost;
4. recuperar o modelo XGBoost do pipeline;
5. utilizar o SHAP para explicar a previsão;
6. ordenar as características pelo impacto absoluto;
7. apresentar inicialmente os 5 fatores de maior impacto.

O fluxo será:

**Cliente** \
↓ \
**10 variáveis originais** \
↓ \
**Preparação + Imputação** \
↓ \
**15 características** \
↓ \
**XGBoost** \
↓ \
**SHAP individual** \
↓ \
**Top 5 fatores da previsão**

O sinal do valor SHAP também possui significado:

- valor SHAP positivo: contribui para aumentar a saída do modelo associada à inadimplência;
- valor SHAP negativo: contribui para reduzir essa saída.

A magnitude indica a intensidade da contribuição daquela característica para a previsão específica.

## 22. Visualização individual com SHAP Waterfall

Após validar numericamente os valores SHAP do cliente, adicionamos uma representação gráfica da explicação.

O gráfico **SHAP Waterfall** mostra como as características individuais do cliente deslocam a saída do modelo em relação ao seu valor de referência.

Cada variável contribui para deslocar a previsão:

- em direção a maior risco de inadimplência; ou
- em direção a menor risco de inadimplência.

A visualização complementa a tabela de valores SHAP e facilita a interpretação da decisão por parte do analista de crédito.

Para manter a aplicação objetiva, serão apresentados os **5 fatores de maior impacto**, enquanto as demais características são agrupadas no gráfico.

## 23. Interpretação do SHAP Waterfall

O gráfico Waterfall permite explicar individualmente a previsão realizada pelo XGBoost.

Para o cliente utilizado no teste, o valor de referência do modelo foi:

**E[f(X)] = -2,694**

Após considerar as características específicas desse cliente, a saída do modelo passou para:

**f(x) = -3,841**

As principais contribuições individuais foram:

- `atrasos_30_59_dias = 0`: contribuição SHAP de aproximadamente -0,29;
- `atrasos_90_mais_dias = 0`: aproximadamente -0,27;
- `linhas_credito_abertas = 5`: aproximadamente -0,17;
- `idade = 35`: aproximadamente +0,16;
- `atrasos_60_89_dias = 0`: aproximadamente -0,12.

As demais características apresentaram contribuição conjunta de aproximadamente -0,46.

Neste cliente, portanto, predominam contribuições que reduzem a saída do modelo associada à inadimplência.

### Escala do gráfico

É importante observar que os valores apresentados no Waterfall não representam diretamente probabilidades.

Para este classificador XGBoost, o TreeSHAP apresenta a explicação na escala da saída bruta do modelo (log-odds).

Assim:

**E[f(X)] → contribuições SHAP → f(x) → transformação logística → probabilidade de inadimplência**

Neste exemplo:

**f(x) ≈ -3,841 → probabilidade ≈ 2,10%**

A aplicação apresenta ao analista a probabilidade, por ser mais intuitiva, enquanto o gráfico SHAP apresenta a decomposição matemática da saída do modelo.

## 24. Validação do cenário de crédito recusado

Após validar o funcionamento da aplicação para um cliente classificado como aprovado, foi realizado um segundo teste com um perfil de maior risco.

O modelo estimou:

**Probabilidade de inadimplência: 71,29%**

Como o limiar operacional definido anteriormente é de:

**10,00%**

temos:

**71,29% > 10,00%**

Portanto, a aplicação classificou o cliente como inadimplente segundo a regra operacional e apresentou:

**CRÉDITO RECUSADO**

### Explicação individual com SHAP

A análise SHAP mostrou que os principais fatores associados à saída do modelo para este cliente foram:

- `atrasos_90_mais_dias = 4`: SHAP ≈ +1,9694;
- `atrasos_60_89_dias = 2`: SHAP ≈ +1,1192;
- `atrasos_30_59_dias = 1`: SHAP ≈ +0,3654;
- `dependentes = 5`: SHAP ≈ +0,1532;
- `financiamentos_imobiliarios = 1`: SHAP ≈ -0,1317.

Neste exemplo, os diferentes tipos de atraso tiveram forte contribuição positiva para a saída do modelo associada à inadimplência, especialmente os atrasos superiores a 90 dias.

O teste confirma que a aplicação funciona nos dois cenários previstos:

1. probabilidade abaixo do limiar → crédito aprovado;
2. probabilidade igual ou superior ao limiar → crédito recusado.

Também foi validado que a explicação SHAP individual funciona nos dois cenários, permitindo ao analista visualizar quais características mais influenciaram cada previsão.

## 25. Do desenvolvimento local ao deploy

Até este momento, a aplicação Streamlit foi executada localmente por meio do comando:

`uv run streamlit run app.py`

Nesse cenário, o Streamlit inicia um servidor local e disponibiliza a aplicação em:

`localhost:8501`

Isso significa que a aplicação está funcionando no computador de desenvolvimento, mas ainda não está publicada para acesso externo.

### O que significa fazer o deploy?

Deploy é o processo de disponibilizar uma aplicação em um ambiente onde ela possa ser executada e acessada pelos usuários.

No nosso projeto, o fluxo passa a ser:

**Código Python**
↓
**Git**
↓
**GitHub**
↓
**Servidor de hospedagem**
↓
**Aplicação Streamlit disponível pela internet**

O ambiente de deploy deverá ser capaz de reconstruir a aplicação utilizando os arquivos versionados no projeto.

Por isso, precisamos garantir que estejam disponíveis:

- `app.py`: interface e lógica da aplicação;
- `src/consumer_credit_risk/`: código de preparação dos dados;
- `pyproject.toml` e `uv.lock`: dependências e ambiente do projeto;
- modelo XGBoost treinado;
- limiar operacional utilizado para a decisão.

### Diferença entre Git, GitHub e Deploy

**Git** controla as versões do projeto localmente.

**GitHub** armazena e compartilha o repositório do projeto remotamente.

**Deploy** executa a aplicação em uma infraestrutura externa, permitindo que ela seja acessada por outros usuários.

Assim, GitHub não é o deploy propriamente dito. Ele funciona como repositório do código que poderá ser utilizado pelo ambiente responsável pela execução da aplicação.